In [ ]:
#!/usr/bin/env python3
"""
Construct BEA benchmark Industry×Industry A matrix (ITA) from Supply-Use tables,
and close the model with respect to households using:
  - Personal Consumption Expenditures (PCE) as household demand
  - Compensation of employees as household income (payments from industries)

Inputs (must exist in working directory):
  - Use_SUT_Detail.xlsx   (sheet "2017")
  - Supply_Detail.xlsx    (sheet "2017")

Outputs:
  - A_ixi_2017.csv
  - A_closed_households_2017.csv
  - multipliers_2017.csv

Key formulas (ITA):
  B = U * inv(diag(x))         (commodities × industries)
  D = V.T * inv(diag(q))       (industries × commodities)
  A = D @ B                    (industries × industries)

Household closure:
  Let:
    c_i   = PCE purchases from industry i   (vector over industries)
    W_j   = compensation paid by industry j (vector over industries)
    X_j   = industry output (column sums of V)
    H     = total household income proxy = sum_j W_j

  Household column (industry i to household):
    a_{i,h} = c_i / H

  Household row (household to industry j):
    a_{h,j} = W_j / X_j

  Bottom-right element a_{h,h} = 0

Then:
  A_closed = [[A, a_{:,h}],
              [a_{h,:}, 0 ]]
"""

from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd


SHEET = "2017"

# BEA detail layout (typical): header codes at row 5, data start at row 6
HEADER_CODE_ROW = 5
DATA_START_ROW = 6

# The row above codes often contains column labels/descriptions
HEADER_LABEL_ROW = 4

# In BEA: col 0 = row code, col 1 = row description
ROW_CODE_COL = 0
ROW_DESC_COL = 1


def _as_str(x) -> str:
    return "" if pd.isna(x) else str(x).strip()


def _is_industry_code(code: str) -> bool:
    """
    Heuristic filter for industry columns:
      - exclude empty
      - exclude final demand codes that start with 'F'
      - exclude totals/controls that start with 'T'
    """
    code = code.strip()
    if not code:
        return False
    if code.startswith("F"):
        return False
    if code.startswith("T"):
        return False
    return True


def read_bea_detail_table(xlsx_path: Path, sheet: str) -> dict:
    """
    Read a BEA-style 'detail' table into structured components:
      - row codes + row descriptions
      - column codes + column descriptions
      - numeric data block (rows: commodities; cols: all coded columns after first 2 cols)
    """
    raw = pd.read_excel(xlsx_path, sheet_name=sheet, header=None)

    col_codes = [_as_str(c) for c in raw.iloc[HEADER_CODE_ROW, :].tolist()]
    col_labels = [_as_str(c) for c in raw.iloc[HEADER_LABEL_ROW, :].tolist()]

    # Data rows start at DATA_START_ROW; capture codes/descriptions
    row_codes_all = raw.iloc[DATA_START_ROW:, ROW_CODE_COL].map(_as_str).tolist()
    row_descs_all = raw.iloc[DATA_START_ROW:, ROW_DESC_COL].map(_as_str).tolist()

    # End of commodity block: stop at first row code that begins with 'T' or 'V'
    end_idx = len(row_codes_all)
    for i, rc in enumerate(row_codes_all):
        if rc.startswith("T") or rc.startswith("V"):
            end_idx = i
            break

    row_codes = row_codes_all[:end_idx]
    row_descs = row_descs_all[:end_idx]

    numeric_block = raw.iloc[DATA_START_ROW : DATA_START_ROW + end_idx, :].copy()

    # Drop first two columns (code+desc), keep the rest
    data = numeric_block.iloc[:, 2:].copy()
    data.columns = col_codes[2:]
    data.index = row_codes

    # numeric coercion
    data = data.apply(pd.to_numeric, errors="coerce").fillna(0.0)

    return {
        "raw": raw,
        "data": data,                    # rows: commodity codes; cols: coded columns (industries + final demand + totals)
        "row_codes": row_codes,
        "row_descs": pd.Series(row_descs, index=row_codes),
        "col_codes": col_codes[2:],
        "col_labels": pd.Series(col_labels[2:], index=col_codes[2:]),
    }


def find_column_by_label(col_labels: pd.Series, patterns: list[str]) -> str:
    """
    Find a column code whose label matches any regex in patterns (case-insensitive).
    Returns column code. Raises if not found.
    """
    labels = col_labels.fillna("")
    for pat in patterns:
        m = labels.str.contains(pat, case=False, regex=True, na=False)
        hits = labels.index[m].tolist()
        if hits:
            return hits[0]
    raise KeyError(f"No column found matching patterns: {patterns}")


def find_row_by_desc(row_descs: pd.Series, patterns: list[str]) -> str:
    """
    Find a row code whose description matches any regex in patterns (case-insensitive).
    Returns row code. Raises if not found.
    """
    desc = row_descs.fillna("")
    for pat in patterns:
        m = desc.str.contains(pat, case=False, regex=True, na=False)
        hits = desc.index[m].tolist()
        if hits:
            return hits[0]
    raise KeyError(f"No row found matching patterns: {patterns}")


def construct_A_ixi_ita(use_tbl: dict, supply_tbl: dict) -> tuple[pd.DataFrame, pd.Series, pd.Series, pd.DataFrame, pd.DataFrame]:
    """
    Construct open I×I A matrix (ITA):
      U: commodities×industries from Use
      V: commodities×industries from Supply
      x: industry output (col sums of V)
      q: commodity output (row sums of V)
      B = U / x
      D = (V / q).T
      A = D @ B
    Returns (A, x, q, B, D)
    """
    U_all: pd.DataFrame = use_tbl["data"]
    V_all: pd.DataFrame = supply_tbl["data"]

    # Industry columns: intersection of codes present in both tables
    U_cols = [c for c in U_all.columns if _is_industry_code(str(c))]
    V_cols = [c for c in V_all.columns if _is_industry_code(str(c))]
    ind_codes = sorted(set(U_cols).intersection(V_cols))
    if not ind_codes:
        raise ValueError("No overlapping industry columns found between Use and Supply after filtering.")

    U = U_all.loc[:, ind_codes].copy()
    V = V_all.loc[:, ind_codes].copy()

    # Align commodities (intersection)
    common_comm = U.index.intersection(V.index)
    if len(common_comm) == 0:
        raise ValueError("No overlapping commodity row codes between Use and Supply.")
    U = U.loc[common_comm, :]
    V = V.loc[common_comm, :]

    # Outputs
    x = V.sum(axis=0)  # industry outputs
    q = V.sum(axis=1)  # commodity outputs

    # Drop zero-output industries/commodities to avoid divide-by-zero
    nonzero_ind = x[x != 0].index
    nonzero_com = q[q != 0].index
    U = U.loc[nonzero_com, nonzero_ind]
    V = V.loc[nonzero_com, nonzero_ind]
    x = x.loc[nonzero_ind]
    q = q.loc[nonzero_com]

    B = U.divide(x, axis=1)           # commodities×industries
    D = V.divide(q, axis=0).T         # industries×commodities
    A = pd.DataFrame(D.to_numpy() @ B.to_numpy(), index=D.index, columns=B.columns)

    return A, x, q, B, D


def close_with_households(
    A: pd.DataFrame,
    x: pd.Series,
    use_tbl: dict,
    *,
    household_name: str = "Households",
) -> tuple[pd.DataFrame, dict]:
    """
    Close A by adding one endogenous household sector using:
      - household demand column from PCE (Use table final demand column)
      - household income row from Compensation of employees (Use table value added rows)

    Returns (A_closed, info_dict).
    """
    U_all: pd.DataFrame = use_tbl["data"]
    col_labels: pd.Series = use_tbl["col_labels"]
    row_descs: pd.Series = use_tbl["row_descs"]

    ind_codes = list(A.columns)

    # --- Find PCE final demand column ---
    # BEA labels vary slightly; these patterns catch the usual cases.
    pce_col = find_column_by_label(
        col_labels,
        patterns=[
            r"personal consumption expenditures",
            r"\bpce\b",
            r"personal consumption",
        ],
    )

    # PCE vector over commodities: U_all[commodity, pce_col]
    # Convert commodity purchases to industry purchases using market shares embedded in A construction?
    # For standard closure in I×I, we need household purchases FROM INDUSTRIES by industry.
    #
    # BEA Use has commodities by industries + final demand columns in commodity space.
    # To get PCE by industry, we can:
    #   1) compute commodity PCE vector y_c (commodities×1)
    #   2) map commodities -> industries using the same D matrix used in ITA (industry shares of commodity output)
    #
    # However we don't have D here unless we pass it; easiest: reconstruct D from Supply via earlier call and pass it in.
    #
    # Alternative pragmatic approach:
    #   Use the published "PCE by industry" if present (rare in this table), or
    #   approximate by distributing commodity PCE to industries by their shares in each commodity (D).
    #
    # We will require D to do this properly. So we’ll recompute D by re-deriving it from the already-built A is not possible.
    #
    # Practical fix: pass D in OR recompute from Supply. Since we only have Use here, we can’t.
    #
    # So: we assume the Use table includes an "industries" block for PCE (i.e., PCE column exists in the same commodity table),
    # and we will convert commodity PCE to industry demand using a *commodity-to-industry share matrix* computed from Supply earlier.
    raise NotImplementedError("Need D (industry-by-commodity market shares) to map commodity PCE to industry purchases.")


def leontief_inverse(A: np.ndarray) -> np.ndarray:
    I = np.eye(A.shape[0])
    return np.linalg.inv(I - A)


def simple_output_multipliers(A: np.ndarray) -> np.ndarray:
    L = leontief_inverse(A)
    return L.sum(axis=0)  # column sums

In [13]:
use_xlsx = Path("../rampr/data/io/Use_SUT_Detail.xlsx")
supply_xlsx = Path("../rampr/data/io/Supply_Detail.xlsx")


use_tbl = read_bea_detail_table(use_xlsx, SHEET)
supply_tbl = read_bea_detail_table(supply_xlsx, SHEET)

A, x, q, B, D = construct_A_ixi_ita(use_tbl, supply_tbl)

# --- Build household closure (using D to map commodity PCE -> industry purchases) ---
col_labels: pd.Series = use_tbl["col_labels"]
row_descs: pd.Series = use_tbl["row_descs"]
U_all: pd.DataFrame = use_tbl["data"]

ind_codes = list(A.columns)

# Find PCE final demand column in Use
pce_col = find_column_by_label(
    col_labels,
    patterns=[r"personal consumption expenditures", r"\bpce\b", r"personal consumption"],
)

# Commodity PCE vector (commodities×1), aligned to commodities in D columns
# D is industries×commodities, with columns = commodity codes used in construction.
comm_codes = list(D.columns)
y_c = U_all.reindex(index=comm_codes).loc[:, pce_col].fillna(0.0).to_numpy().reshape(-1, 1)

# Convert to industry purchases using market shares: y_i = D @ y_c
# Result: industries×1, dollars of PCE attributed to industries.
y_i = (D.to_numpy() @ y_c).reshape(-1)

# Find Compensation of employees row (value added block in Use, not in commodity block).
# IMPORTANT: In BEA detail files, value added rows are *below* the commodity block we read.
# So we must search the raw sheet for a row whose description matches "Compensation of employees"
raw = use_tbl["raw"]

# Scan entire sheet for the compensation row in column 1 (description)
desc_col = raw.iloc[:, ROW_DESC_COL].map(_as_str)
code_col = raw.iloc[:, ROW_CODE_COL].map(_as_str)

comp_row_idx = None
for i in range(len(desc_col)):
    if "compensation of employees" in desc_col.iat[i].lower():
        comp_row_idx = i
        break
if comp_row_idx is None:
    raise KeyError("Could not find 'Compensation of employees' row in Use sheet.")

# Extract compensation payments by industry from that row:
# The industry columns start at col index 2 and align with the coded columns row.
# Build a map from column code -> column index in raw
header_codes_full = raw.iloc[HEADER_CODE_ROW, :].map(_as_str).tolist()
col_index_by_code = {c: j for j, c in enumerate(header_codes_full)}

# Compensation vector over industries (W_j)
W = []
missing = []
for jcode in ind_codes:
    if jcode not in col_index_by_code:
        missing.append(jcode)
        W.append(0.0)
    else:
        val = raw.iat[comp_row_idx, col_index_by_code[jcode]]
        W.append(float(val) if pd.notna(val) else 0.0)
if missing:
    raise KeyError(f"Missing some industry columns when extracting compensation: {missing[:10]} ...")

W = np.array(W, dtype=float)  # dollars paid to households by industry

# Household income proxy: total compensation
H = W.sum()
if H <= 0:
    raise ValueError("Total compensation (household income proxy) is zero or negative; cannot close households.")

# Household column: a_{i,h} = (PCE from industry i) / H
#a_ih = y_i / H  # industries

# Shares of PCE by industry (sum to 1 if total_pce>0)
total_pce = y_i.sum()
if total_pce <= 0:
    raise ValueError("Total mapped PCE is zero; cannot close households.")
pce_shares = y_i / total_pce

MPC = 0.8  # choose 0.6–0.9; 0.8 is a common starting point
a_ih = MPC * pce_shares

# Household row: a_{h,j} = (Comp paid by industry j) / x_j
# x is a Series indexed by industry code
x_vec = x.reindex(ind_codes).to_numpy()
if np.any(x_vec <= 0):
    bad = [ind_codes[k] for k in np.where(x_vec <= 0)[0][:10]]
    raise ValueError(f"Some industries have zero/negative output in x: {bad} ...")
a_hj = W / x_vec  # industries

# --- Two-sector closure: Households + Leakage sink ---
total_pce = y_i.sum()
pce_shares = y_i / total_pce

MPC = 0.6  # try 0.6

a_iH = MPC * pce_shares        # industries -> household column
a_LH = 1.0 - MPC               # leakage -> household column
a_Hj = W / x_vec               # household row payments from industries

n = len(ind_codes)
A_open = A.to_numpy()

A_closed = np.zeros((n + 2, n + 2), dtype=float)
# industries-industries
A_closed[:n, :n] = A_open

# add household column
A_closed[:n, n] = a_iH
A_closed[n, :n] = a_Hj

# leakage sector is a sink: only receives from households
A_closed[n+1, n] = a_LH

idx = ind_codes + ["Households", "Leakage"]
A_closed_df = pd.DataFrame(A_closed, index=idx, columns=idx)


# Write outputs
A.to_csv(f"A_ixi_{SHEET}.csv", float_format="%.10g")
A_closed_df.to_csv(f"A_closed_households_{SHEET}.csv", float_format="%.10g")

# Multipliers
m_type1 = simple_output_multipliers(A_open)
m_type2 = simple_output_multipliers(A_closed)

mult_df = pd.DataFrame(
    {
        "type1_simple_output_multiplier": m_type1,
        "type2_simple_output_multiplier": m_type2[:n],  # industries only
    },
    index=ind_codes,
)
mult_df.to_csv(f"multipliers_{SHEET}.csv", float_format="%.10g")

print(f"Wrote A_ixi_{SHEET}.csv shape={A.shape}")
print(f"Wrote A_closed_households_{SHEET}.csv shape={A_closed_df.shape}")
print(f"Wrote multipliers_{SHEET}.csv shape={mult_df.shape}")
print(f"PCE column used: {pce_col}  label='{use_tbl['col_labels'].get(pce_col, '')}'")
print(f"Compensation row index in sheet: {comp_row_idx}  code='{code_col.iat[comp_row_idx]}'  desc='{desc_col.iat[comp_row_idx]}'")



Wrote A_ixi_2017.csv shape=(401, 401)
Wrote A_closed_households_2017.csv shape=(403, 403)
Wrote multipliers_2017.csv shape=(401, 2)
PCE column used: F01000  label='Personal consumption expenditures'
Compensation row index in sheet: 409  code='V00100'  desc='Compensation of employees'


In [14]:
m_type2.max()

np.float64(12.652856772849397)

In [15]:
m_type2.mean()

np.float64(4.126295219918338)

In [16]:
np.median(m_type2)

np.float64(4.183158472440228)